# Duplicates videos

In [1]:
import pandas as pd
df_raw = pd.read_csv('/Users/van/Desktop/processed_final.csv')
df = df_raw.copy()

In [2]:
# desc = []

# for i, value in enumerate(df['description_json'].iloc[:1000]):
#     try:
#         decoded = json.loads(value)
#         desc.append(decoded['play_by_play'])
#     except Exception as error:
#         print(
#             "Position:", i,
#             "\nIndex:", df.index[i],
#             "\nValue:", repr(value)[:200],
#             "\nError:", type(error).__name__,
#             error
#         )
#         break

In [3]:
import json
desc = [
    json.loads(value).get('play_by_play')
    for value in df['description_json'].iloc[:len(df['description_json'])]]

#desc_df = pd.DataFrame(desc)
#desc_df = desc_df.rename(columns={desc_df.columns[0]: 'text'})


desc_df = (
    pd.DataFrame({'text': desc})
      .dropna(subset=['text'])
      .reset_index(drop=True))
#df.columns.to_list()

In [4]:
desc_df.shape

(12102, 1)

In [5]:
import string
from scipy.stats import fisher_exact
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english') \
                 + ['would', 'one', 'say', 'write', 'subject', 'line', 'go', 'video', 'rugby', 'get', 'know', 'like', 'black', 'yeah',
                    'good', 'really'])

def preprocessing(sentence):
    # standard data cleaning
    for punctuation in string.punctuation:
        sentence = sentence.replace(punctuation, '')
        sentence = ''.join(char for char in sentence if not char.isdigit()) \
            .lower() \
            .strip()

    # removing common words
    word_tokens = word_tokenize(sentence)

    # lemmatizing
    lemmatized_words = [
        WordNetLemmatizer().lemmatize(
            WordNetLemmatizer().lemmatize(word, pos="v"),
            pos="n"
        )
        for word in word_tokens]

    stopwords_removed = [w for w in lemmatized_words if w not in stop_words]

    return ' '.join(stopwords_removed)

In [6]:
desc_df['text_clean']=desc_df['text'].apply(preprocessing)

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer()

vectorized_documents = vectorizer.fit_transform(desc_df['text_clean'])
vectorized_documents = pd.DataFrame(
    vectorized_documents.toarray(),
    columns = vectorizer.get_feature_names_out()
)


# Instantiate the LDA
n_components = 10
lda_model = LatentDirichletAllocation(n_components=n_components, max_iter = 100)

# Fit the LDA on the vectorized documents
lda_model.fit(vectorized_documents)

,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",100
,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [12]:
def print_topics(lda_model, vectorizer, top_words):
    # 1. TOPIC MIXTURE OF WORDS FOR EACH TOPIC
    topic_mixture = pd.DataFrame(
        lda_model.components_,
        columns = vectorizer.get_feature_names_out()
    )

    # 2. FINDING THE TOP WORDS FOR EACH TOPIC
    ## Number of topics
    n_components = topic_mixture.shape[0]

    ## Top words for each topic
    for topic in range(n_components):
        print("-"*10)
        print(f"For topic {topic}, here are the the top {top_words} words with weights:")

        topic_df = topic_mixture.iloc[topic]\
            .sort_values(ascending = False).head(top_words)

        print(round(topic_df,3))

In [13]:
print(print_topics(lda_model,vectorizer, 10))

----------
For topic 0, here are the the top 10 words with weights:
haka           1110.937
perform        1099.559
player          994.489
team            930.857
stand           689.612
chant           683.100
traditional     584.100
new             560.441
zealand         535.358
pitch           461.167
Name: 0, dtype: float64
----------
For topic 1, here are the the top 10 words with weights:
speak        396.612
family       362.880
play         263.263
show         238.352
coach        233.338
interview    224.416
team         209.554
express      205.381
new          200.911
reflect      198.541
Name: 1, dtype: float64
----------
For topic 2, here are the the top 10 words with weights:
savea     846.100
ardie     562.100
ioane     506.100
super     463.752
rieko     389.100
clarke    372.096
chief     350.100
blue      346.861
caleb     339.100
matatū    274.100
Name: 2, dtype: float64
----------
For topic 3, here are the the top 10 words with weights:
play     1160.234
well    

In [12]:
# topic_mixture

In [13]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [14]:
vectorized_example = vectorizer.transform(example)
vectorized_example = pd.DataFrame(
    vectorized_example.toarray(),
    columns = vectorizer.get_feature_names_out()
)
topic_probabilities = lda_model.transform(vectorized_example)

In [15]:
topic_probabilities

array([[0.01827629, 0.01823768, 0.45951634, 0.18356513, 0.32040456]])